In [1]:
!pip install pandas

In [2]:
import pandas as pd

In [5]:
import pandas as pd

row1 = ["kim", 32]
row2 = ["lee", 27] 
row3 = ["park", 45]

colNames = ["name", "age"]


df1 = pd.DataFrame([row1, row2, row3], columns=colNames)

print(df1)

   name  age
0   kim   32
1   lee   27
2  park   45


In [4]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Spark_Study").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/16 08:15:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [6]:
spDf = spark.createDataFrame([row1 , row2, row3], schema=colNames)

In [7]:
spDf.show()

+----+---+
|name|age|
+----+---+
| kim| 32|
| lee| 27|
|park| 45|
+----+---+



In [8]:
spDf2 = spark.createDataFrame(df1)
spDf2.show()

+----+---+
|name|age|
+----+---+
| kim| 32|
| lee| 27|
|park| 45|
+----+---+



In [10]:
print ( type(df1), type(spDf), type(spDf2) )

<class 'pandas.core.frame.DataFrame'> <class 'pyspark.sql.classic.dataframe.DataFrame'> <class 'pyspark.sql.classic.dataframe.DataFrame'>


In [11]:
spDf.createOrReplaceTempView("s_table")

In [12]:
print( spDf)

DataFrame[name: string, age: bigint]


In [15]:
sql = """
select name,age from s_table where age > 30
"""
result = spark.sql(sql)

In [16]:
result.show()

+----+---+
|name|age|
+----+---+
| kim| 32|
|park| 45|
+----+---+



In [17]:
result2 = spDf.filter(spDf.age >= 30)
result2.show()

+----+---+
|name|age|
+----+---+
| kim| 32|
|park| 45|
+----+---+



In [29]:
from pyspark.sql.functions import expr

In [31]:
result3 = spDf.withColumn("newCol", expr("case when age >30 then 'over 30' else 'lower then 30' end"))
result3.show()

+----+---+-------------+
|name|age|       newCol|
+----+---+-------------+
| kim| 32|      over 30|
| lee| 27|lower then 30|
|park| 45|      over 30|
+----+---+-------------+



In [32]:
rdd = spDf.rdd
print(rdd)

MapPartitionsRDD[21] at javaToPython at NativeMethodAccessorImpl.java:0


In [33]:
print (type (rdd))

<class 'pyspark.core.rdd.RDD'>


In [34]:
rdd.collect()

[Row(name='kim', age=32), Row(name='lee', age=27), Row(name='park', age=45)]

In [24]:
cols = spDf.columns

In [25]:
nameCol = cols.index("name")
print(nameCol)

0


In [35]:
result = rdd.filter(lambda x: x[nameCol] == "kim")
result.collect()

[Row(name='kim', age=32)]

In [39]:
def newCol(x):
    oName = x.name
    oAge = x.age
    name = oName +"_님"
    age = oAge * 10
    newCol = f"{name} {age}"
    return (name, age, newCol)

In [40]:
result = rdd.map(lambda x: newCol(x))
result.collect()

[('kim_님', 320, 'kim_님 320'),
 ('lee_님', 270, 'lee_님 270'),
 ('park_님', 450, 'park_님 450')]

In [43]:
spark.stop()